# Prepare Clothing+Beauty Atomic Files


In [1]:
from pathlib import Path
import pandas as pd
import glob
import numpy as np
import torch


In [ ]:
# --- Config ---
BEAUTY_CATEGORY = "Beauty_and_Personal_Care"
CLOTHING_CATEGORY = "Clothing_Shoes_and_Jewelry"
DATA_DIR: str = "../data"

# Data split cutoff dates
TRAIN_END_CUTOFF_DATE: str = "2022-08-01"
VALID_END_CUTOFF_DATE: str = "2022-10-01"

# User & item thresholds
USER_MIN_REVIEWS: int = 5
WARM_USER_MIN_REVIEWS: int = 10
WARM_ITEM_MIN_REVIEWS: int = 5
CLOTHING_MIN_TRAIN_REVIEWS: int = 3

# Downsampling for faster iteration
RANDOM_SEED: int = 42
MAX_TRAIN_SIZE: int | None = None
MAX_VALID_SIZE: int | None = None
MAX_TEST_SIZE: int | None = None

# Image embeddings
EMBEDDINGS_DIR = "../data/embeddings"
CLIP_DIM = 512


## Load reviews


In [3]:
df_reviews = pd.read_csv(f"{DATA_DIR}/reviews.csv")
display(df_reviews.sample(10))
df_reviews.info()


,user_id,parent_asin,rating,timestamp,category
13421291,AHPVQD4W2IIWAKL3AUVYQ63IQSWA,B07QL4MCFH,5.0,1639852324138,Clothing_Shoes_and_Jewelry
20500724,AFRNZCPVL3RMUVO47LGNYFOTMZRQ,B099CJ5F6J,5.0,1658080926428,Beauty_and_Personal_Care
15487760,AHE3CJTWLLDFNY6YZ2JHQKGWGVRA,B07VPNT42Z,5.0,1644916924551,Clothing_Shoes_and_Jewelry
7285441,AHZBWIE6ASWR3DDZINJIG2MF254Q,B08ZDHMR5Q,5.0,1624273742285,Beauty_and_Personal_Care
19968686,AF2GWPKPXN3KSVM7RGXTOD5DAMVA,B08C7W1897,5.0,1656685198227,Beauty_and_Personal_Care
17089069,AF5BKZ3CNVBPDBOTIAHKVKYZEYKA,B07Z928RGZ,5.0,1649275151183,Clothing_Shoes_and_Jewelry
16010899,AFS2D7V6KQMJFXP6ZR5MSRLO5N2A,B017Y8XMB4,2.0,1646353586268,Clothing_Shoes_and_Jewelry
18961789,AEFAACHL5OGNEWI2A2VOVFPDCIYQ,B08VWCMJRM,5.0,1654153024288,Beauty_and_Personal_Care
23544423,AFJO4CKASNWHLC6MUHENNPB4U22Q,B08HHQCTRC,5.0,1665000329735,Beauty_and_Personal_Care
9900201,AFOKG6DT262IPOHK523J4VOM3AAQ,B089M32YWN,5.0,1629915282795,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26876611 entries, 0 to 26876610
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      object 
 1   parent_asin  object 
 2   rating       float64
 3   timestamp    int64  
 4   category     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 1.0+ GB


## Load items


In [4]:
df_items = pd.read_csv(f"{DATA_DIR}/items.csv")
display(df_items.sample(10))
df_items.info()

,parent_asin,title,price,store,category
2007964,B088PDB8R9,Haola Women's Short Sleeve Tops Solid Scoop Ne...,NaN,Haola,Clothing_Shoes_and_Jewelry
681471,B01LKU26AA,Dosoni Women's Winter Fuzzy Slipper Socks Non ...,11.99,Dosoni,Clothing_Shoes_and_Jewelry
2404452,B09YNDBXFK,"Little Girls Sunflower Swimsuit, Girls Long Sl...",NaN,Cozy Feeling,Clothing_Shoes_and_Jewelry
1822221,B07GLN9XTW,AmeriMark Women's Mock Neck Sleeveless Top – T...,NaN,AmeriMark,Clothing_Shoes_and_Jewelry
1208007,B08FPZSTLQ,Comfort Trends Clogs for Women Nurse Shoes - S...,NaN,Comfort Trends,Clothing_Shoes_and_Jewelry
90532,B07QT96YW1,BEAUTIFUL Beaded Hair Barrette with Wood Stick...,30.98,Aptos Trading Company,Beauty_and_Personal_Care
1963673,B0933FMXCM,VALANDY Sports Bra for Women Seamless Light Su...,NaN,VALANDY,Clothing_Shoes_and_Jewelry
2188984,B07VJDFTH7,"Keychains for Women, Flower Crystal Rhinestone...",NaN,JJIA,Clothing_Shoes_and_Jewelry
1719124,B09BL49DQ9,Newborn Baby Boy Summer Clothes Letter Print R...,NaN,itkidboy,Clothing_Shoes_and_Jewelry
1214333,B07VPRWW93,Fantasie Women's Impression Underwire Molded Bra,46.50,FANTASIE,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2694121 entries, 0 to 2694120
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   parent_asin  object 
 1   title        object 
 2   price        float64
 3   store        object 
 4   category     object 
dtypes: float64(1), object(4)
memory usage: 102.8+ MB


# As RecBole expects integer IDs, map string IDs to integers.
# (Item mapping is done later, after the clothing item filter, so iids are dense with no holes.)
user_ids: set[str] = set(df_reviews["user_id"])
user_map: dict[str, int] = {uid: i+1 for i, uid in enumerate(sorted(user_ids))}


In [5]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set(df_reviews["user_id"])
item_ids: set[str] = set(df_reviews["parent_asin"])

user_map: dict[str, int] = {uid: i+1 for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i+1 for i, pid in enumerate(sorted(item_ids))}


In [6]:
df_reviews['uid'] = df_reviews['user_id'].map(user_map)
# iid assignment skipped — done after clothing item filter below


## Filter users by minimum review count (beauty only)


In [7]:
# Valid/test evaluate on beauty — only require minimum reviews in the beauty category.
beauty_mask = df_reviews["category"] == BEAUTY_CATEGORY
before_users = df_reviews.loc[beauty_mask, "uid"].nunique()
beauty_counts = df_reviews[beauty_mask].groupby("uid").size()
valid_users = beauty_counts[beauty_counts >= USER_MIN_REVIEWS].index
df_reviews = df_reviews[df_reviews["uid"].isin(valid_users)]
after_users = df_reviews.loc[df_reviews["category"] == BEAUTY_CATEGORY, "uid"].nunique()
print(f"Removed {before_users - after_users:,} users with < {USER_MIN_REVIEWS} beauty reviews "
      f"({after_users:,} users remain)")


Removed 4,221,111 users with < 5 beauty reviews (168,377 users remain)


## Split and filter train/valid/test 


In [8]:
def _date_to_ms(date_str: str) -> int:
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

df_beauty_reviews = df_reviews[df_reviews["category"] == BEAUTY_CATEGORY]
timestamps = sorted(df_beauty_reviews["timestamp"].values)
train_start_ts = timestamps[0]
train_end_ts = _date_to_ms(TRAIN_END_CUTOFF_DATE)
valid_end_ts = _date_to_ms(VALID_END_CUTOFF_DATE)
print(f"Train start timestamp: {train_start_ts} ({pd.Timestamp(train_start_ts, unit='ms', tz='UTC')})")
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")

Train start timestamp: 1609459264769 (2021-01-01 00:01:04.769000+00:00)
Train end timestamp: 1659312000000 (2022-08-01 00:00:00+00:00)
Valid end timestamp: 1664582400000 (2022-10-01 00:00:00+00:00)


In [9]:
# Split beauty reviews by time
df_beauty_train = df_beauty_reviews[df_beauty_reviews["timestamp"] <= train_end_ts]
df_beauty_valid = df_beauty_reviews[(df_beauty_reviews["timestamp"] > train_end_ts) & (df_beauty_reviews["timestamp"] <= valid_end_ts)]
df_beauty_test = df_beauty_reviews[df_beauty_reviews["timestamp"] > valid_end_ts]

beauty_train_user_ids = set(df_beauty_train["uid"].unique())

# Train = beauty_train + all clothing reviews (additional signal from the same users)
clothing_reviews = df_reviews[
    (df_reviews["category"] == CLOTHING_CATEGORY)
    & (df_reviews["uid"].isin(beauty_train_user_ids))
]
df_train = pd.concat([df_beauty_train, clothing_reviews], ignore_index=True)

df_valid = df_beauty_valid.copy()
df_test = df_beauty_test.copy()

print(f"Train reviews: {len(df_train):,}  (beauty: {len(df_beauty_train):,}, clothing: {len(clothing_reviews):,})")
print('Train interactions per user', len(df_train) / len(df_train['uid'].unique()))

if MAX_TRAIN_SIZE is not None and len(df_train) > MAX_TRAIN_SIZE:
    df_train = df_train.sample(n=MAX_TRAIN_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled train reviews to {len(df_train):,}")
if MAX_VALID_SIZE is not None and len(df_valid) > MAX_VALID_SIZE:
    df_valid = df_valid.sample(n=MAX_VALID_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled valid reviews to {len(df_valid):,}")
if MAX_TEST_SIZE is not None and len(df_test) > MAX_TEST_SIZE:
    df_test = df_test.sample(n=MAX_TEST_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled test reviews to {len(df_test):,}")

df_valid = df_valid[df_valid["uid"].isin(beauty_train_user_ids)]
df_test = df_test[df_test["uid"].isin(beauty_train_user_ids)]

# --- Filter: drop tail clothing items with < CLOTHING_MIN_TRAIN_REVIEWS train reviews ---
clothing_mask = df_train["category"] == CLOTHING_CATEGORY
clothing_counts = df_train[clothing_mask].groupby("parent_asin").size()
valid_clothing_pids = set(clothing_counts[clothing_counts >= CLOTHING_MIN_TRAIN_REVIEWS].index)
n_before = len(clothing_counts)
df_train = df_train[~clothing_mask | df_train["parent_asin"].isin(valid_clothing_pids)]
n_after = df_train[df_train["category"] == CLOTHING_CATEGORY]["parent_asin"].nunique()
print(f"Filtered clothing items: {n_before:,} -> {n_after:,} "
      f"(removed {n_before - n_after:,} with < {CLOTHING_MIN_TRAIN_REVIEWS} train reviews)")

# --- Map item IDs to integers (after filtering, so iids are dense with no holes) ---
all_pids = set(df_train["parent_asin"]) | set(df_valid["parent_asin"]) | set(df_test["parent_asin"])
item_map: dict[str, int] = {pid: i+1 for i, pid in enumerate(sorted(all_pids))}
df_train["iid"] = df_train["parent_asin"].map(item_map)
df_valid["iid"] = df_valid["parent_asin"].map(item_map)
df_test["iid"] = df_test["parent_asin"].map(item_map)
df_items["iid"] = df_items["parent_asin"].map(item_map)
df_items = df_items.dropna(subset=["iid"])
df_items["iid"] = df_items["iid"].astype(int)
df_all = pd.concat([df_train, df_valid, df_test])

print(f"Valid reviews: {len(df_valid):,}")
print(f"Test reviews: {len(df_test):,}")

display(pd.DataFrame({
    "split": ["train", "valid", "test"],
    "reviews": [len(df_train), len(df_valid), len(df_test)],
    "users": [df_train["uid"].nunique(), df_valid["uid"].nunique(), df_test["uid"].nunique()],
    "items": [df_train["iid"].nunique(), df_valid["iid"].nunique(), df_test["iid"].nunique()],
}))


Train reviews: 2,391,297  (beauty: 1,125,023, clothing: 1,266,274)
Train interactions per user 15.281122393553458
Filtered clothing items: 435,032 -> 102,005 (removed 333,027 with < 3 train reviews)
Valid reviews: 139,338
Test reviews: 197,706


,split,reviews,users,items
0,train,2001046,156487,317363
1,valid,139338,47209,52826
2,test,197706,55594,66743


## Define cold vs. warm users/items with train data


In [10]:
beauty_train_counts = df_beauty_train.groupby("uid").size()
cold_user_ids: set[int] = set(beauty_train_counts[beauty_train_counts < WARM_USER_MIN_REVIEWS].index)
warm_user_ids: set[int] = set(beauty_train_counts[beauty_train_counts >= WARM_USER_MIN_REVIEWS].index)
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} beauty train reviews): {len(warm_user_ids):,}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} beauty train reviews): {len(cold_user_ids):,}")


Warm users (>= 10 beauty train reviews): 20,100
Cold users (< 10 beauty train reviews): 136,387


In [11]:
## Define cold vs. warm items with train data
df_train_items = df_train.groupby("iid").size().reset_index(name="num_train")
cold_item_ids: set[int] = set(df_train_items[df_train_items["num_train"] < WARM_ITEM_MIN_REVIEWS]["iid"])
warm_item_ids: set[int] = set(df_train_items[df_train_items["num_train"] >= WARM_ITEM_MIN_REVIEWS]["iid"])
print(f"Warm items (>= {WARM_ITEM_MIN_REVIEWS} train reviews): {len(warm_item_ids):,}")
print(f"Cold items (< {WARM_ITEM_MIN_REVIEWS} train reviews): {len(cold_item_ids):,}")


Warm items (>= 5 train reviews): 108,941
Cold items (< 5 train reviews): 208,422


In [12]:
all_user_ids = set(df_all['uid'].unique())
all_item_ids = df_items['iid'].isin(df_all['iid'].unique())
print(f"Unique users in all splits: {len(all_user_ids):,}")
print(f"Unique items in all splits: {all_item_ids.sum():,}")


Unique users in all splits: 156,487
Unique items in all splits: 352,857


## Write atomic files


In [13]:
def get_user_category(uid: int) -> int:
    if uid in warm_user_ids:
        return 0  # Warm user
    return 1      # Cold user

def write_user_file(path: Path, uids: set[int]) -> None:
    rows = [(uid, get_user_category(uid)) for uid in uids]
    df_out = pd.DataFrame(rows, columns=["user_id:token", "cold:float"])
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_item_file(path: Path, df_items_subset: pd.DataFrame) -> None:
    df_out = df_items_subset[["iid", "title", "store", "price"]].copy()
    df_out["title"] = df_out["title"].fillna("").astype(str).str.replace('"', "", regex=False)
    df_out["store"] = df_out["store"].fillna("").astype(str).str.replace('"', "", regex=False)
    df_out["price"] = pd.to_numeric(df_out["price"], errors="coerce").fillna("")
    df_out["cold"] = df_out["iid"].apply(lambda iid: 1 if iid not in warm_item_ids else 0)
    df_out["target"] = df_items_subset["category"].apply(
        lambda c: 1.0 if c == BEAUTY_CATEGORY else 0.0
    )
    df_out.columns = ["item_id:token", "title:token", "store:token", "price:float", "cold:float", "target:float"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_inter_file(path: Path, df: pd.DataFrame) -> None:
    df_out = df[["uid", "iid", "rating", "timestamp"]].copy()
    df_out.columns = ["user_id:token", "item_id:token", "rating:float", "timestamp:float"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")


In [14]:
dataset_prefix = Path(DATA_DIR) / "clothing-beauty" / "clothing-beauty"
dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(dataset_prefix.with_suffix(".train.inter"), df_train)
write_inter_file(dataset_prefix.with_suffix(".valid.inter"), df_valid)
write_inter_file(dataset_prefix.with_suffix(".test.inter"), df_test)
write_user_file(dataset_prefix.with_suffix(".user"), all_user_ids)
write_item_file(dataset_prefix.with_suffix(".item"), df_items[all_item_ids])


Wrote ../data/clothing-beauty/clothing-beauty.train.inter (2,001,046 rows)
Wrote ../data/clothing-beauty/clothing-beauty.valid.inter (139,338 rows)
Wrote ../data/clothing-beauty/clothing-beauty.test.inter (197,706 rows)
Wrote ../data/clothing-beauty/clothing-beauty.user (156,487 rows)
Wrote ../data/clothing-beauty/clothing-beauty.item (352,857 rows)


## Save image embeddings


In [15]:
shard_paths = sorted(glob.glob(f"{EMBEDDINGS_DIR}/*.parquet"))
print(f"Found {len(shard_paths)} CLIP embedding shard(s)")

dfs = [pd.read_parquet(p) for p in shard_paths]
df_clip = pd.concat(dfs, ignore_index=True).drop_duplicates(subset="parent_asin", keep="last")
print(f"Loaded {len(df_clip):,} CLIP image embeddings (deduplicated by parent_asin)")

asin_to_clip: dict[str, np.ndarray] = dict(zip(
    df_clip["parent_asin"],
    df_clip["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)),
))


Found 914 CLIP embedding shard(s)
Loaded 4,380,169 CLIP image embeddings (deduplicated by parent_asin)


In [17]:
dataset_items = df_items[all_item_ids].copy()
n_items = len(dataset_items)
embs = np.zeros((n_items, CLIP_DIM), dtype=np.float32)
iid_to_idx: dict[int, int] = {}

n_found = 0
for idx, (_, row) in enumerate(dataset_items.iterrows()):
    iid = int(row["iid"])
    vec = asin_to_clip.get(row["parent_asin"])
    if vec is not None:
        embs[idx] = vec
        n_found += 1
    iid_to_idx[iid] = idx

torch.save(
    {"embeddings": torch.from_numpy(embs), "iid_to_idx": iid_to_idx},
    f"{DATA_DIR}/clothing-beauty/clip_image_embeddings.pt",
)
print(f"Saved CLIP image embeddings: {n_found:,} / {n_items:,} items have images")


Saved CLIP image embeddings: 352,818 / 352,857 items have images
